# Symmetry Property Matrix Report

This notebook evaluates **all four properties** for both symmetry sets (`G` and `SO(3)`), for each block (`encoder`, `equivariant_conv`, `upsampler`, `attention`, `decoder`):

- LEFT_INVARIANCE
- RIGHT_INVARIANCE
- LEFT_EQUIVARIANCE
- RIGHT_EQUIVARIANCE

For each check it reports:
- relative error (`rel`)
- RMS error (`rms`)
- tolerance thresholds
- pass/fail status


In [20]:
from __future__ import annotations

from contextlib import contextmanager
from pathlib import Path
import os
import sys

import pandas as pd
import torch

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from models.SR_double_conv_SRattn import (
    AttentionBlock,
    CubochoricOptimizingLocalIsoDecoder,
    EquivariantSpatialConv,
    EquivariantTransposeConv,
    LocalIsoCrystalEncoder,
)
import models.SR_double_conv_SRattn_a1 as sr_a1

from symmetry_tests._symmetry_utils import (
    choose_group_symmetry,
    choose_so3_probe_quaternion,
    error_metrics,
    feature_action_matrix,
    left_action_quaternions,
    make_block_distance_matrix,
    passive_right_action_for_active_left,
    quat_conjugate,
    quat_mul,
    quaternion_error_metrics,
    random_unit_quats,
    right_action_quaternions,
    to_passive_quaternions,
)

print(f"Using repo root: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"Pandas: {pd.__version__}")


Using repo root: /home/warren/projects/Reynolds-QSR
PyTorch: 2.5.1
Pandas: 2.3.2


In [21]:
def pass_rule(rel: float, rms: float, rel_tol: float, rms_tol: float) -> bool:
    return bool(rel <= rel_tol or rms <= rms_tol)

def add_result(
    rows: list[dict],
    *,
    block: str,
    crystal: str,
    side: str,
    group_name: str,
    property_kind: str,
    rel: float,
    rms: float,
    rel_tol: float,
    rms_tol: float,
    detail: str = "",
) -> None:
    rows.append(
        {
            "block": block,
            "crystal": crystal,
            "side": side.upper(),
            "group": group_name,
            "property": f"{side.upper()}_{property_kind}_{group_name}",
            "rel": float(rel),
            "rms": float(rms),
            "rel_tol": float(rel_tol),
            "rms_tol": float(rms_tol),
            "passed": pass_rule(float(rel), float(rms), float(rel_tol), float(rms_tol)),
            "detail": detail,
        }
    )

def run_attention_delta(block: AttentionBlock, feat: torch.Tensor, *, h: int, w: int, block_h: int, block_w: int) -> torch.Tensor:
    d_block = make_block_distance_matrix(block_h, block_w, dtype=feat.dtype, device=feat.device)
    return block(feat.unsqueeze(0), d_block, h, w, block_h, block_w).squeeze(0)

def transform_active_input(q_active: torch.Tensor, sym: torch.Tensor, side: str) -> torch.Tensor:
    side_k = str(side).lower()
    if side_k == "left":
        return left_action_quaternions(q_active, sym)
    if side_k == "right":
        return right_action_quaternions(q_active, sym)
    raise ValueError(f"Unknown side: {side}")

def passive_left_action_for_active_right(q_passive: torch.Tensor, sym: torch.Tensor) -> torch.Tensor:
    # active right: q_a -> q_a ⊗ r  implies passive: q_p -> r^{-1} ⊗ q_p
    sym_inv = quat_conjugate(sym.view(1, 4))[0]
    sym_inv_batch = sym_inv.view(1, 4).expand(q_passive.shape[0], 4)
    return quat_mul(sym_inv_batch, q_passive)

@contextmanager
def patched_sampler(table_quats_passive: torch.Tensor):
    original = sr_a1._sample_fz_quaternions_passive

    def fake_sample(
        group_name: str,
        resolution: int,
        method: str,
        dtype: torch.dtype,
        device: torch.device,
        max_rows: int | None = None,
    ) -> torch.Tensor:
        del group_name, resolution, method
        out = table_quats_passive.to(device=device, dtype=dtype)
        if max_rows is not None:
            out = out[: int(max_rows)]
        return out

    sr_a1._sample_fz_quaternions_passive = fake_sample
    try:
        yield
    finally:
        sr_a1._sample_fz_quaternions_passive = original

def build_decoder_without_orix_sampling(crystal: str):
    encoder = LocalIsoCrystalEncoder(crystal=crystal, dtype=torch.float32, device="cpu").eval()
    g = choose_group_symmetry(encoder.sym_ops)
    so3_probe = choose_so3_probe_quaternion(encoder.sym_ops)
    q_base_active = random_unit_quats(32, seed=606, dtype=encoder.embedding.group_mats.dtype, device=encoder.embedding.group_mats.device)
    q_table_active = torch.cat([q_base_active, left_action_quaternions(q_base_active, g), left_action_quaternions(q_base_active, so3_probe)], dim=0)
    q_table_passive = to_passive_quaternions(q_table_active)
    with patched_sampler(q_table_passive):
        decoder = CubochoricOptimizingLocalIsoDecoder(
            encoder=encoder,
            cubochoric_resolution=1,
            method="cubochoric",
            num_starts=1,
            steps=0,
            lr=0.05,
            target_irreps="a1",
            max_table_rows=None,
            table_cache_dir=None,
        ).eval()
    return encoder, decoder, g, so3_probe, q_base_active

def calibrate_feature_action(base_feat: torch.Tensor, trans_feat: torch.Tensor, irreps, sym: torch.Tensor):
    mats = {
        "I": torch.eye(base_feat.shape[-1], device=base_feat.device, dtype=base_feat.dtype),
        "D": feature_action_matrix(irreps, sym, device=base_feat.device, dtype=base_feat.dtype, variant="D"),
        "D_T": feature_action_matrix(irreps, sym, device=base_feat.device, dtype=base_feat.dtype, variant="D_T"),
    }
    best_name = None
    best_mat = None
    best_rel = float("inf")
    best_rms = float("inf")
    for name, mat in mats.items():
        rel, rms = error_metrics(base_feat @ mat, trans_feat)
        if rel < best_rel or (abs(rel - best_rel) < 1e-12 and rms < best_rms):
            best_name, best_mat, best_rel, best_rms = name, mat, rel, rms
    return best_name, best_mat, best_rel, best_rms

def block_tolerances(block: str) -> tuple[float, float]:
    # Keep one tolerance pair per block family for a consistent pass rule.
    if block == "encoder":
        return 4e-4, 4e-5
    if block == "equivariant_conv":
        return 9e-4, 9e-5
    if block == "upsampler":
        return 1.2e-3, 1.2e-4
    if block == "attention":
        return 1.3e-3, 1.3e-4
    if block == "decoder":
        return 8e-4, 8e-4
    raise ValueError(block)


In [22]:
rows: list[dict] = []

for crystal in ("fcc", "hcp"):
    encoder = LocalIsoCrystalEncoder(crystal=crystal, dtype=torch.float32, device="cpu").eval()
    device = encoder.embedding.group_mats.device
    dtype = encoder.embedding.group_mats.dtype
    g = choose_group_symmetry(encoder.sym_ops)
    so3_probe = choose_so3_probe_quaternion(encoder.sym_ops)

    # Shared base input for feature-output blocks.
    q_base_feature = random_unit_quats(96, seed=707, dtype=dtype, device=device)

    # Build isolated feature-output block functions.
    conv = EquivariantSpatialConv(kernel_size=3, irreps_in=encoder.irreps_a1, irreps_out=encoder.irreps_a1, use_residual=True).to(device=device, dtype=dtype).eval()
    up = EquivariantTransposeConv(kernel_size=3, upsample_factor=2, use_residual=True, irreps_in=encoder.irreps_a1, irreps_out=encoder.irreps_a1).to(device=device, dtype=dtype).eval()
    attn = AttentionBlock(irreps_feat=encoder.irreps_a1, num_channels=2).to(device=device, dtype=dtype).eval()
    with torch.no_grad():
        torch.manual_seed(17)
        attn.lin_out.weight.normal_(mean=0.0, std=0.05)
        attn.pos_bias.weight.fill_(0.10)
        attn.pos_bias.bias.fill_(0.01)

    def encoder_fn(q_active: torch.Tensor) -> torch.Tensor:
        return encoder.forward_a1_active(q_active)

    def conv_fn(q_active: torch.Tensor) -> torch.Tensor:
        feat = encoder.forward_a1_active(q_active)
        return conv(feat, (8, 12))  # 96 = 8*12

    def up_fn(q_active: torch.Tensor) -> torch.Tensor:
        feat = encoder.forward_a1_active(q_active)
        y, _ = up(feat, (8, 12))
        return y

    def attn_fn(q_active: torch.Tensor) -> torch.Tensor:
        feat = encoder.forward_a1_active(q_active)
        return run_attention_delta(attn, feat, h=8, w=12, block_h=4, block_w=4)

    feature_blocks = {
        "encoder": encoder_fn,
        "equivariant_conv": conv_fn,
        "upsampler": up_fn,
        "attention": attn_fn,
    }

    # Evaluate feature blocks for all side/group combinations.
    for block_name, block_fn in feature_blocks.items():
        rel_tol, rms_tol = block_tolerances(block_name)
        for side in ("left", "right"):
            for group_name, sym in (("G", g), ("SO3", so3_probe)):
                q_t = transform_active_input(q_base_feature, sym, side)
                with torch.no_grad():
                    y_base = block_fn(q_base_feature)
                    y_t = block_fn(q_t)
                    feat_base = encoder.forward_a1_active(q_base_feature)
                    feat_t = encoder.forward_a1_active(q_t)

                # Invariance metric.
                rel_inv, rms_inv = error_metrics(y_base, y_t)
                add_result(
                    rows,
                    block=block_name,
                    crystal=crystal,
                    side=side,
                    group_name=group_name,
                    property_kind="INVARIANCE",
                    rel=rel_inv,
                    rms=rms_inv,
                    rel_tol=rel_tol,
                    rms_tol=rms_tol,
                )

                # Equivariance metric using calibrated feature action (I / D / D^T).
                action_name, action_mat, cal_rel, cal_rms = calibrate_feature_action(
                    feat_base, feat_t, encoder.irreps_a1, sym
                )
                rel_eq, rms_eq = error_metrics(y_base @ action_mat, y_t)
                add_result(
                    rows,
                    block=block_name,
                    crystal=crystal,
                    side=side,
                    group_name=group_name,
                    property_kind="EQUIVARIANCE",
                    rel=rel_eq,
                    rms=rms_eq,
                    rel_tol=rel_tol,
                    rms_tol=rms_tol,
                    detail=f"feature_action={action_name};cal_rel={cal_rel:.3e};cal_rms={cal_rms:.3e}",
                )

    # Decoder block uses a dedicated deterministic lookup table setup.
    enc_dec, decoder, g_dec, so3_dec, q_base_dec = build_decoder_without_orix_sampling(crystal)
    rel_tol_dec, rms_tol_dec = block_tolerances("decoder")

    for side in ("left", "right"):
        for group_name, sym in (("G", g_dec), ("SO3", so3_dec)):
            q_t_active = transform_active_input(q_base_dec, sym, side)

            q_base_passive = to_passive_quaternions(q_base_dec)
            q_t_passive = to_passive_quaternions(q_t_active)

            with torch.no_grad():
                feat_base = enc_dec.forward_a1(q_base_passive)
                feat_t = enc_dec.forward_a1(q_t_passive)
                q_dec_base = decoder(feat_base)
                q_dec_t = decoder(feat_t)

            # Invariance metric in quaternion space (sign-aligned).
            rel_inv, rms_inv = quaternion_error_metrics(q_dec_base, q_dec_t)
            add_result(
                rows,
                block="decoder",
                crystal=crystal,
                side=side,
                group_name=group_name,
                property_kind="INVARIANCE",
                rel=rel_inv,
                rms=rms_inv,
                rel_tol=rel_tol_dec,
                rms_tol=rms_tol_dec,
            )

            # Equivariance expectation differs by action side in passive convention.
            if side == "left":
                q_expected = passive_right_action_for_active_left(q_dec_base, sym)
            else:
                q_expected = passive_left_action_for_active_right(q_dec_base, sym)

            rel_eq, rms_eq = quaternion_error_metrics(q_expected, q_dec_t)
            add_result(
                rows,
                block="decoder",
                crystal=crystal,
                side=side,
                group_name=group_name,
                property_kind="EQUIVARIANCE",
                rel=rel_eq,
                rms=rms_eq,
                rel_tol=rel_tol_dec,
                rms_tol=rms_tol_dec,
            )


In [23]:
df = pd.DataFrame(rows)

side_order = {"LEFT": 0, "RIGHT": 1}
group_order = {"G": 0, "SO3": 1}
prop_order = {"INVARIANCE": 0, "EQUIVARIANCE": 1}
block_order = {"encoder": 0, "equivariant_conv": 1, "upsampler": 2, "attention": 3, "decoder": 4}

def split_prop(p: str):
    # PROPERTY format: SIDE_KIND_GROUP
    parts = p.split("_")
    side = parts[0]
    kind = parts[1]
    group = parts[-1]
    return side, kind, group

tmp = df["property"].apply(split_prop)
df["_side"] = tmp.apply(lambda x: x[0])
df["_kind"] = tmp.apply(lambda x: x[1])
df["_group"] = tmp.apply(lambda x: x[2])

df["_block_order"] = df["block"].map(block_order)
df["_side_order"] = df["_side"].map(side_order)
df["_group_order"] = df["_group"].map(group_order)
df["_kind_order"] = df["_kind"].map(prop_order)

df = df.sort_values(["crystal", "_block_order", "_side_order", "_group_order", "_kind_order"]).reset_index(drop=True)

overall = bool(df["passed"].all())
print(f"Overall pass: {overall}")

summary = (
    df.groupby(["crystal", "block"], as_index=False)["passed"]
    .all()
    .rename(columns={"passed": "all_checks_passed"})
)
print("Per-block summary")
display(summary)

matrix_summary = (
    df.groupby(["crystal", "block", "side", "group", "_kind"], as_index=False)["passed"]
    .all()
    .rename(columns={"_kind": "kind", "passed": "all_passed"})
)
print("\nProperty matrix summary")
display(matrix_summary)

drop_cols = ["_side", "_kind", "_group", "_block_order", "_side_order", "_group_order", "_kind_order"]
df_out = df.drop(columns=drop_cols)
preferred_cols = ["block", "property", "passed", "rms", "rms_tol"]
ordered_cols = preferred_cols + [c for c in df_out.columns if c not in preferred_cols]
df_out = df_out[ordered_cols]
print("\nDetailed results")
display(df_out)

out_csv = ROOT / "out" / "symmetry_test_notebook_results.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_csv, index=False)
print(f"Saved detailed CSV: {out_csv}")

# Separate per-crystal exports.
for crystal_name in ("fcc", "hcp"):
    crystal_csv = ROOT / "out" / f"symmetry_test_notebook_results_{crystal_name}.csv"
    df_out[df_out["crystal"] == crystal_name].to_csv(crystal_csv, index=False)
    print(f"Saved {crystal_name.upper()} CSV: {crystal_csv}")


Overall pass: False
Per-block summary


,crystal,block,all_checks_passed
0,fcc,attention,False
1,fcc,decoder,False
2,fcc,encoder,False
3,fcc,equivariant_conv,False
4,fcc,upsampler,False
5,hcp,attention,False
6,hcp,decoder,False
7,hcp,encoder,False
8,hcp,equivariant_conv,False
9,hcp,upsampler,False



Property matrix summary


,crystal,block,side,group,kind,all_passed
0,fcc,attention,LEFT,G,EQUIVARIANCE,True
1,fcc,attention,LEFT,G,INVARIANCE,False
2,fcc,attention,LEFT,SO3,EQUIVARIANCE,True
3,fcc,attention,LEFT,SO3,INVARIANCE,False
4,fcc,attention,RIGHT,G,EQUIVARIANCE,True
...,...,...,...,...,...,...
75,hcp,upsampler,LEFT,SO3,INVARIANCE,False
76,hcp,upsampler,RIGHT,G,EQUIVARIANCE,True
77,hcp,upsampler,RIGHT,G,INVARIANCE,True
78,hcp,upsampler,RIGHT,SO3,EQUIVARIANCE,False



Detailed results


,block,property,passed,rms,rms_tol,crystal,side,group,rel,rel_tol,detail
0,encoder,LEFT_INVARIANCE_G,False,1.703138e-01,0.00004,fcc,LEFT,G,1.319244e+00,0.0004,
1,encoder,LEFT_EQUIVARIANCE_G,True,2.286169e-07,0.00004,fcc,LEFT,G,1.770858e-06,0.0004,feature_action=D;cal_rel=1.771e-06;cal_rms=2.2...
2,encoder,LEFT_INVARIANCE_SO3,False,1.966092e-01,0.00004,fcc,LEFT,SO3,1.522928e+00,0.0004,
3,encoder,LEFT_EQUIVARIANCE_SO3,True,3.887534e-07,0.00004,fcc,LEFT,SO3,3.011270e-06,0.0004,feature_action=D_T;cal_rel=3.011e-06;cal_rms=3...
4,encoder,RIGHT_INVARIANCE_G,True,5.354786e-08,0.00004,fcc,RIGHT,G,4.147797e-07,0.0004,
...,...,...,...,...,...,...,...,...,...,...,...
75,decoder,LEFT_EQUIVARIANCE_SO3,True,1.899961e-08,0.00080,hcp,LEFT,SO3,3.799921e-08,0.0008,
76,decoder,RIGHT_INVARIANCE_G,True,0.000000e+00,0.00080,hcp,RIGHT,G,0.000000e+00,0.0008,
77,decoder,RIGHT_EQUIVARIANCE_G,False,7.071068e-01,0.00080,hcp,RIGHT,G,1.414214e+00,0.0008,
78,decoder,RIGHT_INVARIANCE_SO3,False,5.576677e-01,0.00080,hcp,RIGHT,SO3,1.115335e+00,0.0008,


Saved detailed CSV: /home/warren/projects/Reynolds-QSR/out/symmetry_test_notebook_results.csv
Saved FCC CSV: /home/warren/projects/Reynolds-QSR/out/symmetry_test_notebook_results_fcc.csv
Saved HCP CSV: /home/warren/projects/Reynolds-QSR/out/symmetry_test_notebook_results_hcp.csv


In [24]:
df = pd.DataFrame(rows)

side_order = {"LEFT": 0, "RIGHT": 1}
group_order = {"G": 0, "SO3": 1}
prop_order = {"INVARIANCE": 0, "EQUIVARIANCE": 1}
block_order = {"encoder": 0, "equivariant_conv": 1, "upsampler": 2, "attention": 3, "decoder": 4}

def split_prop(p: str):
    # PROPERTY format: SIDE_KIND_GROUP
    parts = p.split("_")
    side = parts[0]
    kind = parts[1]
    group = parts[-1]
    return side, kind, group

tmp = df["property"].apply(split_prop)
df["_side"] = tmp.apply(lambda x: x[0])
df["_kind"] = tmp.apply(lambda x: x[1])
df["_group"] = tmp.apply(lambda x: x[2])

df["_block_order"] = df["block"].map(block_order)
df["_side_order"] = df["_side"].map(side_order)
df["_group_order"] = df["_group"].map(group_order)
df["_kind_order"] = df["_kind"].map(prop_order)

df = df.sort_values(["crystal", "_block_order", "_side_order", "_group_order", "_kind_order"]).reset_index(drop=True)

overall = bool(df["passed"].all())
print(f"Overall pass: {overall}")

summary = (
    df.groupby(["crystal", "block"], as_index=False)["passed"]
    .all()
    .rename(columns={"passed": "all_checks_passed"})
)
print("Per-block summary")
display(summary)

matrix_summary = (
    df.groupby(["crystal", "block", "side", "group", "_kind"], as_index=False)["passed"]
    .all()
    .rename(columns={"_kind": "kind", "passed": "all_passed"})
)
print("\nProperty matrix summary")
display(matrix_summary)

drop_cols = ["_side", "_kind", "_group", "_block_order", "_side_order", "_group_order", "_kind_order"]
df_out = df.drop(columns=drop_cols)
preferred_cols = ["block", "property", "passed", "rms", "rms_tol"]
ordered_cols = preferred_cols + [c for c in df_out.columns if c not in preferred_cols]
df_out = df_out[ordered_cols]
print("\nDetailed results")
display(df_out)

out_csv = ROOT / "out" / "symmetry_test_notebook_results.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_csv, index=False)
print(f"Saved detailed CSV: {out_csv}")

# Separate per-crystal exports.
for crystal_name in ("fcc", "hcp"):
    crystal_csv = ROOT / "out" / f"symmetry_test_notebook_results_{crystal_name}.csv"
    df_out[df_out["crystal"] == crystal_name].to_csv(crystal_csv, index=False)
    print(f"Saved {crystal_name.upper()} CSV: {crystal_csv}")


Overall pass: False
Per-block summary


,crystal,block,all_checks_passed
0,fcc,attention,False
1,fcc,decoder,False
2,fcc,encoder,False
3,fcc,equivariant_conv,False
4,fcc,upsampler,False
5,hcp,attention,False
6,hcp,decoder,False
7,hcp,encoder,False
8,hcp,equivariant_conv,False
9,hcp,upsampler,False



Property matrix summary


,crystal,block,side,group,kind,all_passed
0,fcc,attention,LEFT,G,EQUIVARIANCE,True
1,fcc,attention,LEFT,G,INVARIANCE,False
2,fcc,attention,LEFT,SO3,EQUIVARIANCE,True
3,fcc,attention,LEFT,SO3,INVARIANCE,False
4,fcc,attention,RIGHT,G,EQUIVARIANCE,True
...,...,...,...,...,...,...
75,hcp,upsampler,LEFT,SO3,INVARIANCE,False
76,hcp,upsampler,RIGHT,G,EQUIVARIANCE,True
77,hcp,upsampler,RIGHT,G,INVARIANCE,True
78,hcp,upsampler,RIGHT,SO3,EQUIVARIANCE,False



Detailed results


,block,property,passed,rms,rms_tol,crystal,side,group,rel,rel_tol,detail
0,encoder,LEFT_INVARIANCE_G,False,1.703138e-01,0.00004,fcc,LEFT,G,1.319244e+00,0.0004,
1,encoder,LEFT_EQUIVARIANCE_G,True,2.286169e-07,0.00004,fcc,LEFT,G,1.770858e-06,0.0004,feature_action=D;cal_rel=1.771e-06;cal_rms=2.2...
2,encoder,LEFT_INVARIANCE_SO3,False,1.966092e-01,0.00004,fcc,LEFT,SO3,1.522928e+00,0.0004,
3,encoder,LEFT_EQUIVARIANCE_SO3,True,3.887534e-07,0.00004,fcc,LEFT,SO3,3.011270e-06,0.0004,feature_action=D_T;cal_rel=3.011e-06;cal_rms=3...
4,encoder,RIGHT_INVARIANCE_G,True,5.354786e-08,0.00004,fcc,RIGHT,G,4.147797e-07,0.0004,
...,...,...,...,...,...,...,...,...,...,...,...
75,decoder,LEFT_EQUIVARIANCE_SO3,True,1.899961e-08,0.00080,hcp,LEFT,SO3,3.799921e-08,0.0008,
76,decoder,RIGHT_INVARIANCE_G,True,0.000000e+00,0.00080,hcp,RIGHT,G,0.000000e+00,0.0008,
77,decoder,RIGHT_EQUIVARIANCE_G,False,7.071068e-01,0.00080,hcp,RIGHT,G,1.414214e+00,0.0008,
78,decoder,RIGHT_INVARIANCE_SO3,False,5.576677e-01,0.00080,hcp,RIGHT,SO3,1.115335e+00,0.0008,


Saved detailed CSV: /home/warren/projects/Reynolds-QSR/out/symmetry_test_notebook_results.csv
Saved FCC CSV: /home/warren/projects/Reynolds-QSR/out/symmetry_test_notebook_results_fcc.csv
Saved HCP CSV: /home/warren/projects/Reynolds-QSR/out/symmetry_test_notebook_results_hcp.csv
